End-to-end Bronze, Silver, and Gold reconciliation

In [0]:

# 1. DEFINE DELTA STORAGE PATHS


bronze_path = "/Volumes/workspace/default/railway_bronze/data"

silver_clean_path = "/Volumes/workspace/default/railway_silver/clean"
silver_quarantine_path = "/Volumes/workspace/default/railway_silver/quarantine"
silver_review_path = "/Volumes/workspace/default/railway_silver/review"
silver_transformed_path = "/Volumes/workspace/default/railway_silver/transformed"

gold_base_path = "/Volumes/workspace/default/railway_gold"



# 2. LOAD BRONZE AND SILVER DATASETS


bronze_df = spark.read.format("delta").load(bronze_path)

clean_df = spark.read.format("delta").load(silver_clean_path)

quarantine_df = spark.read.format("delta").load(silver_quarantine_path)

review_df = spark.read.format("delta").load(silver_review_path)

transformed_df = spark.read.format("delta").load(silver_transformed_path)



# 3. COUNT RECORDS


bronze_count = bronze_df.count()

clean_count = clean_df.count()

quarantine_count = quarantine_df.count()

review_count = review_df.count()

transformed_count = transformed_df.count()


# ------------------------------------------
# 4. LOAD INDEPENDENT DUPLICATE AUDIT
# ------------------------------------------

# Path to the verified duplicate audit dataset
duplicate_audit_path = (
    "/Volumes/workspace/default/"
    "railway_silver/duplicate_audit"
)

# Read the independently generated audit
duplicate_audit_df = (
    spark.read
    .format("delta")
    .load(duplicate_audit_path)
)

# Audit contains one record for the current batch
assert duplicate_audit_df.count() == 1, (
    "Expected exactly one duplicate audit record"
)

# Retrieve the verified duplicate count
duplicate_audit_record = duplicate_audit_df.first()

removed_duplicate_count = (
    duplicate_audit_record["duplicates_removed"]
)

# Verify the audit belongs to the current data snapshot
assert (
    duplicate_audit_record["bronze_records"] == bronze_count
    and duplicate_audit_record["silver_clean_records"] == clean_count
    and duplicate_audit_record["quarantine_records"] == quarantine_count
    and duplicate_audit_record["review_records"] == review_count
), "Duplicate audit counts do not match current datasets"

print("Verified Duplicates Removed:", removed_duplicate_count)


# ------------------------------------------
# 5. INDEPENDENT BRONZE RECONCILIATION
# ------------------------------------------

# Reconcile using the independently audited count
reconciled_count = (
    clean_count
    + quarantine_count
    + review_count
    + removed_duplicate_count
)

# Fail if any Bronze records are unaccounted for
assert bronze_count == reconciled_count, (
    f"Bronze reconciliation failed: "
    f"Bronze={bronze_count}, "
    f"Reconciled={reconciled_count}"
)

print("Independent Bronze Reconciliation: PASSED")


# 6. SILVER TRANSFORMATION VALIDATION


assert clean_count == transformed_count, (
    f"Silver transformation mismatch: "
    f"Clean={clean_count}, Transformed={transformed_count}"
)



# 7. VALIDATE GOLD DATASETS


gold_dataset_names = [
    "overall_kpis",
    "station_performance",
    "train_performance",
    "monthly_performance",
    "yearly_performance",
    "station_recovery"
]

for dataset_name in gold_dataset_names:

    # Read the Gold Delta dataset
    gold_df = (
        spark.read
        .format("delta")
        .load(f"{gold_base_path}/{dataset_name}")
    )

    # Count records
    gold_count = gold_df.count()

    # Check dataset is not empty
    assert gold_count > 0, (
        f"Gold dataset is empty: {dataset_name}"
    )

    print(
        f"Gold: {dataset_name} | "
        f"Records: {gold_count} | PASSED"
    )


# ------------------------------------------
# 8. FINAL VALIDATION SUMMARY
# ------------------------------------------

print("\n========== PIPELINE RECONCILIATION ==========")

print("Bronze Records:", bronze_count)

print("Silver Clean Records:", clean_count)

print("Silver Quarantine Records:", quarantine_count)

print("Silver Review Records:", review_count)

print("Calculated Unaccounted Records:", removed_duplicate_count)

print("Reconciled Records:", reconciled_count)

print("Transformed Silver Records:", transformed_count)

print("\nBRONZE RECORD COUNT RECONCILIATION: PASSED")

print("SILVER TRANSFORMATION RECONCILIATION: PASSED")

print("GOLD DATASET ACCESSIBILITY: PASSED")


Silver Data Integrity Validation

In [0]:


from pyspark.sql import functions as F

# Use the transformed Silver dataset loaded in Cell 1
df = transformed_df

# ------------------------------------------
# 1. CHECK MANDATORY COLUMNS
# ------------------------------------------

mandatory_columns = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Count records with missing mandatory identifiers
missing_ids = df.filter(
    F.col("date").isNull()
    | F.col("train_no").isNull()
    | F.col("station_code").isNull()
    | F.col("seq").isNull()
).count()

assert missing_ids == 0, (
    f"Missing mandatory identifiers: {missing_ids}"
)

print("Mandatory Identifiers: PASSED")


# ------------------------------------------
# 2. CHECK POSITIVE SEQUENCE NUMBERS
# ------------------------------------------

# Sequence numbers must be greater than zero
invalid_seq = df.filter(
    F.col("seq") <= 0
).count()

assert invalid_seq == 0, (
    f"Invalid sequence numbers: {invalid_seq}"
)

print("Sequence Number Validation: PASSED")


# ------------------------------------------
# 3. CHECK DUPLICATE MOVEMENT KEYS
# ------------------------------------------

# Candidate key identified during data quality analysis
movement_keys = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Find keys appearing more than once
duplicate_keys = (
    df.groupBy(*movement_keys)
      .count()
      .filter(F.col("count") > 1)
      .count()
)

assert duplicate_keys == 0, (
    f"Duplicate movement keys: {duplicate_keys}"
)

print("Movement Key Uniqueness: PASSED")


# ------------------------------------------
# 4. CHECK DELAY CATEGORIES
# ------------------------------------------

# Allowed delay classifications
valid_categories = [
    "EARLY",
    "ON_TIME",
    "MINOR_DELAY",
    "MODERATE_DELAY",
    "SEVERE_DELAY",
    "UNKNOWN"
]

# Find unexpected arrival delay categories
invalid_arrival_categories = df.filter(
    F.col("arrival_delay_category").isNull()
    | ~F.col("arrival_delay_category").isin(valid_categories)
).count()

# Find unexpected departure delay categories
invalid_departure_categories = df.filter(
    F.col("departure_delay_category").isNull()
    | ~F.col("departure_delay_category").isin(valid_categories)
).count()

assert invalid_arrival_categories == 0, (
    "Invalid arrival delay categories found"
)

assert invalid_departure_categories == 0, (
    "Invalid departure delay categories found"
)

print("Delay Category Validation: PASSED")


# ------------------------------------------
# 5. CHECK RECOVERY CALCULATION
# ------------------------------------------

# Expected recovery = arrival delay - departure delay
# Compare only rows where both delays are available
recovery_mismatches = df.filter(
    F.col("calculated_arrival_delay_min").isNotNull()
    & F.col("calculated_departure_delay_min").isNotNull()
    & (
        F.abs(
            (
                F.col("calculated_arrival_delay_min")
                - F.col("calculated_departure_delay_min")
            )
            - F.col("delay_recovery_min")
        ) > 0.001
    )
).count()

assert recovery_mismatches == 0, (
    f"Recovery calculation mismatches: {recovery_mismatches}"
)

print("Delay Recovery Calculation: PASSED")


# ------------------------------------------
# 6. CHECK RECORD COUNT
# ------------------------------------------

silver_count = df.count()

assert silver_count == clean_df.count(), (
    "Transformed Silver record count mismatch"
)

print("Silver Record Count: PASSED")


# ------------------------------------------
# FINAL VALIDATION SUMMARY
# ------------------------------------------

print("\n========== SILVER VALIDATION ==========")
print("Total Records:", silver_count)
print("Missing Identifiers:", missing_ids)
print("Invalid Sequence Numbers:", invalid_seq)
print("Duplicate Movement Keys:", duplicate_keys)
print("Invalid Arrival Categories:", invalid_arrival_categories)
print("Invalid Departure Categories:", invalid_departure_categories)
print("Recovery Calculation Mismatches:", recovery_mismatches)

print("\nALL SILVER DATA INTEGRITY CHECKS PASSED")

Gold Aggregation Integrity Validation

In [0]:


from pyspark.sql import functions as F

# 1. LOAD GOLD DATASETS


gold_base_path = "/Volumes/workspace/default/railway_gold"

gold_names = [
    "overall_kpis",
    "station_performance",
    "train_performance",
    "monthly_performance",
    "yearly_performance",
    "station_recovery"
]

gold_dfs = {}

for name in gold_names:

    # Read each Gold Delta dataset
    gold_dfs[name] = (
        spark.read
        .format("delta")
        .load(f"{gold_base_path}/{name}")
    )

# Use transformed Silver from Cell 1
silver_df = transformed_df
silver_count = silver_df.count()



# 2. VALIDATE GOLD AGGREGATION TOTALS


# Every grouping must account for all Silver records
aggregate_names = [
    "station_performance",
    "train_performance",
    "monthly_performance",
    "yearly_performance",
    "station_recovery"
]

for name in aggregate_names:

    # Sum the observations stored in Gold
    gold_total = (
        gold_dfs[name]
        .agg(F.sum("total_observations").alias("total"))
        .first()["total"]
    )

    # Compare Gold with transformed Silver
    assert gold_total == silver_count, (
        f"{name}: Gold={gold_total}, "
        f"Silver={silver_count}"
    )

    print(f"{name}: {gold_total} = {silver_count} | PASSED")



# 3. VALIDATE OVERALL KPI COUNTS


# Overall KPI must contain exactly one row
overall_df = gold_dfs["overall_kpis"]

assert overall_df.count() == 1, (
    "Overall KPI dataset must contain exactly one row"
)

overall = overall_df.first()

# Independently calculate Silver counts
silver_train_count = (
    silver_df.select("train_no").distinct().count()
)

silver_station_count = (
    silver_df.select("station_code").distinct().count()
)

# Compare Gold KPI values with Silver
assert overall["total_observations"] == silver_count
assert overall["total_trains"] == silver_train_count
assert overall["total_stations"] == silver_station_count

print("Overall KPI Counts: PASSED")



# 4. VALIDATE AVERAGE DELAYS


# Recalculate average delays from verified Silver rows
silver_averages = silver_df.agg(

    F.round(
        F.avg(
            F.when(
                F.col("arrival_delay_quality") == "MATCH",
                F.col("calculated_arrival_delay_min")
            )
        ), 2
    ).alias("avg_arrival"),

    F.round(
        F.avg(
            F.when(
                F.col("departure_delay_quality") == "MATCH",
                F.col("calculated_departure_delay_min")
            )
        ), 2
    ).alias("avg_departure")

).first()

# Compare with saved Gold averages
assert (
    overall["avg_arrival_delay_min"]
    == silver_averages["avg_arrival"]
), "Arrival average mismatch"

assert (
    overall["avg_departure_delay_min"]
    == silver_averages["avg_departure"]
), "Departure average mismatch"

print("Overall Average Delays: PASSED")



# 5. VALIDATE GOLD GROUP COUNTS


# Check distinct business keys against Gold row counts
group_checks = {
    "station_performance": "station_code",
    "train_performance": "train_no",
    "station_recovery": "station_code"
}

for name, key in group_checks.items():

    expected = silver_df.select(key).distinct().count()
    actual = gold_dfs[name].count()

    assert actual == expected, (
        f"{name}: Expected {expected}, got {actual}"
    )

    # Ensure no duplicate Gold business keys
    duplicate_groups = (
        gold_dfs[name]
        .groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_groups == 0, (
        f"{name}: Duplicate Gold keys found"
    )

    print(f"{name} Group Uniqueness: PASSED")



# FINAL SUMMARY


print("\n========== GOLD VALIDATION ==========")
print("Silver Records:", silver_count)
print("Distinct Trains:", silver_train_count)
print("Distinct Stations:", silver_station_count)
print("Gold Datasets Validated:", len(gold_names))

print("\nALL GOLD AGGREGATION VALIDATIONS PASSED")

Generate and save the pipeline validation report

In [0]:



from pyspark.sql import functions as F

# ------------------------------------------
# 1. DEFINE AUDIT STORAGE PATHS
# ------------------------------------------

audit_path = (
    "/Volumes/workspace/default/"
    "railway_gold/validation_audit"
)

duplicate_audit_path = (
    "/Volumes/workspace/default/"
    "railway_silver/duplicate_audit"
)


# ------------------------------------------
# 2. READ CURRENT RECORD COUNTS
# ------------------------------------------

bronze_count = bronze_df.count()
clean_count = clean_df.count()
quarantine_count = quarantine_df.count()
review_count = review_df.count()
transformed_count = transformed_df.count()


# ------------------------------------------
# 3. LOAD VERIFIED DUPLICATE AUDIT
# ------------------------------------------

duplicate_audit_df = (
    spark.read
    .format("delta")
    .load(duplicate_audit_path)
)

# The current-batch audit must contain one row
assert duplicate_audit_df.count() == 1, (
    "Expected exactly one duplicate audit record"
)

duplicate_audit_record = duplicate_audit_df.first()

# Read independently verified duplicate count
removed_duplicate_count = int(
    duplicate_audit_record["duplicates_removed"]
)

# Verify audit counts match current datasets
duplicate_audit_passed = all([
    duplicate_audit_record["bronze_records"] == bronze_count,
    duplicate_audit_record["silver_clean_records"] == clean_count,
    duplicate_audit_record["quarantine_records"] == quarantine_count,
    duplicate_audit_record["review_records"] == review_count,
    removed_duplicate_count >= 0
])

print("Verified Duplicates Removed:", removed_duplicate_count)
print("Duplicate Audit Matched:", duplicate_audit_passed)


# ------------------------------------------
# 4. BRONZE TO SILVER RECONCILIATION
# ------------------------------------------

reconciled_count = (
    clean_count
    + quarantine_count
    + review_count
    + removed_duplicate_count
)

bronze_reconciliation_passed = (
    bronze_count == reconciled_count
)


# ------------------------------------------
# 5. SILVER TRANSFORMATION VALIDATION
# ------------------------------------------

silver_reconciliation_passed = (
    clean_count == transformed_count
)


# ------------------------------------------
# 6. GOLD AGGREGATION VALIDATION
# ------------------------------------------

gold_aggregation_passed = True

for name in [
    "station_performance",
    "train_performance",
    "monthly_performance",
    "yearly_performance",
    "station_recovery"
]:

    # Sum Gold observations
    gold_total = (
        gold_dfs[name]
        .agg(
            F.sum("total_observations").alias("total")
        )
        .first()["total"]
    )

    # Each aggregation should account for Silver
    if gold_total != transformed_count:
        gold_aggregation_passed = False


# ------------------------------------------
# 7. OVERALL KPI VALIDATION
# ------------------------------------------

overall_kpi_passed = (
    gold_dfs["overall_kpis"]
    .first()["total_observations"]
    == transformed_count
)


# ------------------------------------------
# 8. COMBINE VALIDATION RESULTS
# ------------------------------------------

all_checks_passed = all([
    duplicate_audit_passed,
    bronze_reconciliation_passed,
    silver_reconciliation_passed,
    gold_aggregation_passed,
    overall_kpi_passed
])


# ------------------------------------------
# 9. PREPARE FINAL AUDIT RECORD
# ------------------------------------------

audit_records = [(
    "RailPulse_India",
    bronze_count,
    clean_count,
    quarantine_count,
    review_count,
    removed_duplicate_count,
    transformed_count,
    len(gold_dfs),
    "PASSED" if all_checks_passed else "FAILED"
)]

audit_columns = [
    "pipeline_name",
    "bronze_records",
    "silver_clean_records",
    "quarantine_records",
    "review_records",
    "duplicates_removed",
    "transformed_silver_records",
    "gold_dataset_count",
    "validation_status"
]

audit_df = spark.createDataFrame(
    audit_records,
    audit_columns
)

# Add execution timestamp
audit_df = audit_df.withColumn(
    "validation_timestamp",
    F.current_timestamp()
)


# ------------------------------------------
# 10. SAVE AUDIT TO DELTA
# ------------------------------------------

# Append preserves historical audit records
audit_df.write \
    .format("delta") \
    .mode("append") \
    .save(audit_path)


# ------------------------------------------
# 11. DISPLAY RECENT AUDIT RECORDS
# ------------------------------------------

audit_check = (
    spark.read
    .format("delta")
    .load(audit_path)
)

display(
    audit_check.orderBy(
        F.col("validation_timestamp").desc()
    ).limit(5)
)


# ------------------------------------------
# 12. FINAL VALIDATION STATUS
# ------------------------------------------

assert all_checks_passed, (
    "PIPELINE VALIDATION FAILED"
)

print("\nFINAL PIPELINE VALIDATION: PASSED")
print("Pipeline: RailPulse_India")
print("Bronze Records:", bronze_count)
print("Verified Duplicates Removed:", removed_duplicate_count)
print("Transformed Silver Records:", transformed_count)
print("Gold Datasets:", len(gold_dfs))
print("Audit Report:", audit_path)
